# 5. Full English translation

Translate the complete Ukrainian source into English and preserve both aligned
versions. Use the same translation prompt and recorded Codex-output format as
in notebook 4. A configured provider factory is an alternative execution path.
Set its actual provider, model and settings in `translator` before using that path.

The final corpus is under annotation. Leave the input paths unset until the
source and its generation manifest are available. This notebook can be inspected
and tested without that final corpus.

In [ ]:
import json
import tempfile
from pathlib import Path

from IPython.display import display
from pandas import DataFrame

from fairforget.dataset_workflow import find_project, load_inputs
from fairforget.io import read_rows, write_rows

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = Path(tempfile.mkdtemp(prefix="warbias-dataset-"))

In [ ]:
from fairforget.dataset_workflow import preview_translation, record_translations
from fairforget.translation.warbias import align_languages, validate_production_source

translator = json.loads((project / "configs/translation/codex.json").read_text())
UKRAINIAN_PATH = None
PRODUCTION_MANIFEST_PATH = None
ASSISTANT_TRANSLATIONS_PATH = None
PROVIDER_FACTORY = None
RUN_TRANSLATION = False

result = {
    "stage": "full_english_translation",
    "status": "waiting_for_ukrainian_input",
    "source_language": "uk",
    "target_language": "en",
    "translation_model": translator["model"],
    "annotation_status": protocol["full_ukrainian_annotation"]["status"],
    "execution_requested": RUN_TRANSLATION,
}
if UKRAINIAN_PATH is not None:
    if PRODUCTION_MANIFEST_PATH is None:
        raise ValueError("Supply the complete source generation manifest")
    ukrainian = read_rows(UKRAINIAN_PATH)
    production = json.loads(Path(PRODUCTION_MANIFEST_PATH).read_text())
    validate_production_source(ukrainian, production, translator)
    display(DataFrame(ukrainian)[["id", "stereotype", "counter_stereotype", "unrelated"]].head())
    result = {
        "stage": "full_english_translation",
        **preview_translation(
            project,
            ukrainian,
            work / "translation-preview",
            config=translator,
            production_manifest=production,
        ),
        "execution_requested": RUN_TRANSLATION,
    }
if RUN_TRANSLATION:
    if UKRAINIAN_PATH is None:
        raise ValueError("Supply the full Ukrainian input before execution")
    if ASSISTANT_TRANSLATIONS_PATH is not None:
        english = record_translations(
            project,
            ukrainian,
            read_rows(ASSISTANT_TRANSLATIONS_PATH),
            work / "translation",
            config=translator,
            production_manifest=production,
            source_origin={
                "kind": "full_ukrainian_generation",
                "manifest_sha256": production["manifest_sha256"],
            },
        )
    elif PROVIDER_FACTORY:
        from fairforget.cli_support import load_factory
        from fairforget.translation.warbias import translate_production

        provider = load_factory(PROVIDER_FACTORY)(config=translator)
        english = translate_production(
            ukrainian,
            production,
            translator,
            project / "prompts/current/translation.txt",
            work / "translation",
            provider=provider,
            execute=True,
        )
    else:
        raise ValueError("Supply recorded assistant translations or a provider factory")
    align_languages(ukrainian, english)
    display(DataFrame(english)[["id", "stereotype", "counter_stereotype", "unrelated"]].head())
    write_rows(work / "translation/ukrainian.jsonl", ukrainian)
    write_rows(work / "translation/english.jsonl", english)
    result = {"stage": "full_english_translation", "status": "translated", "pairs": len(english)}
result